In [2]:
PROJECT_ID = "project-b38b370e-25fb-420a-a54"
BUCKET_URI = "gs://mlops-iris-csv-pipeline-unique"
LOCATION = "us-central1"

print("PROJECT_ID:", PROJECT_ID)
print("BUCKET_URI:", BUCKET_URI)
print("LOCATION:", LOCATION)

PROJECT_ID: project-b38b370e-25fb-420a-a54
BUCKET_URI: gs://mlops-iris-csv-pipeline-unique
LOCATION: us-central1


In [4]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://mlops-iris-csv-pipeline-unique/...


In [5]:
! gsutil cp data/iris.csv {BUCKET_URI}/data/iris.csv

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://data/iris.csv [Content-Type=text/csv]...
/ [1 files][  3.8 KiB/  3.8 KiB]                                                
Operation completed over 1 objects/3.8 KiB.                                      


In [8]:
import sys
!{sys.executable} -m pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 97.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
import joblib
import os
from datetime import datetime


# Read data FROM GCS
data = pd.read_csv(f'{BUCKET_URI}/data/iris.csv')
print("Data loaded! Shape:", data.shape)

Data loaded! Shape: (150, 5)


In [4]:
data.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:
print(data.dtypes)

sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species             str
dtype: object


In [6]:
from sklearn.model_selection import train_test_split

# Features (all columns except target)
X = data.drop('species', axis=1)

# Target column
y = data['species']

# 80% training, 20% evaluation
X_train, X_eval, y_train, y_eval = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Evaluation samples:", len(X_eval))

Training samples: 120
Evaluation samples: 30


In [7]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [8]:
from datetime import datetime
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(timestamp, exist_ok=True)

print("Folder created:", timestamp)

Folder created: 20260618_111618


In [9]:
import joblib

model_path = f"{timestamp}/model.joblib"

joblib.dump(model, model_path)

print("Model saved:", model_path)

Model saved: 20260618_111618/model.joblib


In [10]:
!gsutil cp {model_path} {BUCKET_URI}/artifacts/{timestamp}/

I0618 11:16:40.635143    3290 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://20260618_111618/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  3.4 KiB/  3.4 KiB]                                                
Operation completed over 1 objects/3.4 KiB.                                      
